In [ ]:
# Copyright (c) 2025 Microsoft Corporation.
import sys

sys.path.insert(1, "../../../")

In [ ]:
import json
import logging
import os

import pandas as pd
from pydantic import SecretStr

from benchmark_qed.autod.data_processor.embedding import TextEmbedder
from benchmark_qed.autod.io.text_unit import load_text_units
from benchmark_qed.autod.sampler.sample_gen import acreate_clustered_sample
from benchmark_qed.autoq.io.question import load_questions, save_questions
from benchmark_qed.config.llm_config import LLMConfig, LLMProvider
from benchmark_qed.llm.factory import ModelFactory

logging.basicConfig(level=logging.INFO)
logging.getLogger("httpx").setLevel(logging.ERROR)

In [ ]:
%load_ext dotenv
%dotenv

# Experiment 11a: Hierarchical Chunking — Small Retrieval Chunks

This notebook implements **Experiment 11a**, which evaluates RAG question generation using a hierarchical chunking strategy with small retrieval-level chunks (300 tokens). In hierarchical chunking, documents are split into fine-grained chunks for precise semantic retrieval, where each small chunk retains a reference to its larger parent context used for answer generation.

This experiment serves as the **fine-grained baseline** in the 11a/11b/11c series:

| Experiment | Chunk Size | Description |
|------------|-----------|-------------|
| **11a** (this notebook) | 300 tokens | Small retrieval chunks — high precision, narrow context |
| 11b | 600 tokens | Medium retrieval chunks — balanced precision and context |
| 11c | 1200 tokens | Large retrieval chunks — broad context, lower precision |

**Dataset:** Behind the Tech Podcast Transcripts (70 episodes). Podcast transcripts are long-form documents well-suited for hierarchical chunking experiments, as they contain extended discussions that benefit from fine-grained retrieval.

**Pipeline:** AutoD (data preparation) → AutoQ (data-local + data-global question generation)

## Configs

In [ ]:
# DATA CONFIGS
INPUT_DATA_PATH = "../../datasets/podcast/raw_data"
OUTPUT_DATA_PATH = "./output/podcast/11a/processed_data"
OUTPUT_QUESTIONS_PATH = "./output/podcast/11a/questions"
TEXT_COLUMN = "text"
METADATA_COLUMNS = ["title"]
FILE_ENCODING = "utf-8"

# EXPERIMENT 11a: small retrieval-level chunk size
ENCODING_MODEL = "o200k_base"
CHUNK_SIZE = 300      # small chunks for fine-grained semantic retrieval
CHUNK_OVERLAP = 50    # overlap to preserve context across chunk boundaries

# DATA SAMPLING CONFIGS
# The final sample size will be NUM_CLUSTERS * NUM_SAMPLES_PER_CLUSTER
NUM_CLUSTERS = 20
NUM_SAMPLES_PER_CLUSTER = 5
RANDOM_SEED = 42

# QUESTION GENERATION CONFIGS
NUM_QUESTIONS = 10
OVERSAMPLE_FACTOR = 2.0
CONCURRENT_REQUESTS = 8

# MODEL CONFIGS
API_KEY = SecretStr(os.getenv("OPENAI_API_KEY", ""))
EMBEDDING_MODEL = "text-embedding-3-large"
LLM_MODEL = "gpt-4.1"
LLM_PARAMS = {
    "temperature": 0.0,
    "seed": 42,
}  # adjust this based on your model. For example, some reasoning models do not support temperature settings

In [ ]:
# Set up the embedding model
text_embedder = TextEmbedder(
    ModelFactory.create_embedding_model(
        LLMConfig(
            model=EMBEDDING_MODEL,
            api_key=API_KEY,
            llm_provider=LLMProvider.OpenAIEmbedding,
        )
    )
)

# Set up the LLM
llm = ModelFactory.create_chat_model(
    model_config=LLMConfig(
        model=LLM_MODEL,
        api_key=API_KEY,
        llm_provider=LLMProvider.OpenAIChat,
        call_args=LLM_PARAMS,
    )
)

## Step 1: Data Preparation (AutoD)

Load podcast transcripts, split them into small retrieval-level chunks (300 tokens with 50-token overlap), and embed each chunk for semantic search. With small chunks, each chunk covers a narrow segment of the conversation, enabling precise retrieval of specific passages.

In [ ]:
# Load documents, create small retrieval chunks, embed, and sample
# Experiment 11a uses CHUNK_SIZE=300 (small retrieval chunks)
clustered_sample = await acreate_clustered_sample(
    input_path=INPUT_DATA_PATH,
    output_path=OUTPUT_DATA_PATH,
    text_embedder=text_embedder,
    num_clusters=NUM_CLUSTERS,
    num_samples_per_cluster=NUM_SAMPLES_PER_CLUSTER,
    input_type="text",
    text_tag=TEXT_COLUMN,
    metadata_tags=METADATA_COLUMNS,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    file_encoding=FILE_ENCODING,
    token_encoding=ENCODING_MODEL,
    random_seed=RANDOM_SEED,
)
print(
    f"Loaded {len(clustered_sample.documents)} podcast episode transcripts."
)
print(
    f"Created {len(clustered_sample.text_units)} text units "
    f"(chunk_size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP})."
)
print(
    f"Sampled {len(clustered_sample.sample_texts)} text units "
    f"from {NUM_CLUSTERS} clusters."
)

## Step 2: Question Generation (AutoQ)

Generate data-local and data-global questions grounded in the sampled podcast content.

With small retrieval chunks (300 tokens), data-local questions are tightly focused on specific passages within an episode, while data-global questions require synthesizing information across many such small chunks.

### Data-Local Questions

Data-local questions target specific details found in individual text chunks.

In [ ]:
from benchmark_qed.autoq.config import (
    AssertionConfig,
    GlobalAssertionConfig,
    LocalAssertionConfig,
)
from benchmark_qed.autoq.question_gen.data_questions.local_question_gen import (
    DataLocalQuestionGen,
)

# Load sampled text units
# If you have previously run the data preparation step, you can load from disk:
# sample_texts_df = pd.read_parquet(f"{OUTPUT_DATA_PATH}/sample_texts.parquet")
# sample_texts = load_text_units(df=sample_texts_df)
sample_texts = clustered_sample.sample_texts

local_assertion_config = LocalAssertionConfig()
global_assertion_config = GlobalAssertionConfig()
assertion_config = AssertionConfig(
    local=local_assertion_config,
    **{
        "global": global_assertion_config
    },  # Use dict unpacking since "global" is a keyword
)

data_local_generator = DataLocalQuestionGen(
    llm=llm,
    llm_params=LLM_PARAMS,
    text_embedder=text_embedder,
    text_units=sample_texts,
    concurrent_coroutines=CONCURRENT_REQUESTS,
    random_seed=RANDOM_SEED,
    assertion_config=assertion_config,
)

data_local_results = await data_local_generator.agenerate(
    num_questions=NUM_QUESTIONS,
    oversample_factor=OVERSAMPLE_FACTOR,
)

# Save both candidate and selected questions
save_questions(
    data_local_results.selected_questions,
    f"{OUTPUT_QUESTIONS_PATH}/data_local_questions/",
    "selected_questions",
)
save_questions(
    data_local_results.selected_questions,
    f"{OUTPUT_QUESTIONS_PATH}/data_local_questions/",
    "selected_questions_text",
    question_text_only=True,
)
save_questions(
    data_local_results.candidate_questions,
    f"{OUTPUT_QUESTIONS_PATH}/data_local_questions/",
    "candidate_questions",
)

print(f"Generated {len(data_local_results.selected_questions)} data-local questions")
pd.DataFrame(
    [{"question": q.text, "type": str(q.question_type)} for q in data_local_results.selected_questions]
).head()

### Data-Global Questions

Data-global questions require reasoning over multiple chunks spanning the full dataset. With fine-grained chunks, answering global questions requires integrating many individually narrow pieces of information.

In [ ]:
from benchmark_qed.autoq.question_gen.data_questions.global_question_gen import (
    DataGlobalQuestionGen,
)

# Load all candidate local questions (a larger pool improves global question diversity)
# If you have previously run the data-local step, you can load from disk:
# local_questions = load_questions(
#     f"{OUTPUT_QUESTIONS_PATH}/data_local_questions/candidate_questions.json"
# )
local_questions = data_local_results.candidate_questions
print(f"Using {len(local_questions)} candidate local questions for global generation.")

data_global_generator = DataGlobalQuestionGen(
    llm=llm,
    llm_params=LLM_PARAMS,
    text_embedder=text_embedder,
    local_questions=local_questions,
    concurrent_coroutines=CONCURRENT_REQUESTS,
    random_seed=RANDOM_SEED,
    assertion_config=assertion_config,
)

data_global_results = await data_global_generator.agenerate(
    num_questions=NUM_QUESTIONS,
    oversample_factor=OVERSAMPLE_FACTOR,
)

# Save both candidate and selected questions
save_questions(
    data_global_results.selected_questions,
    f"{OUTPUT_QUESTIONS_PATH}/data_global_questions/",
    "selected_questions",
)
save_questions(
    data_global_results.selected_questions,
    f"{OUTPUT_QUESTIONS_PATH}/data_global_questions/",
    "selected_questions_text",
    question_text_only=True,
)
save_questions(
    data_global_results.candidate_questions,
    f"{OUTPUT_QUESTIONS_PATH}/data_global_questions/",
    "candidate_questions",
)

print(f"Generated {len(data_global_results.selected_questions)} data-global questions")
pd.DataFrame(
    [{"question": q.text, "type": str(q.question_type)} for q in data_global_results.selected_questions]
).head()

## Results Summary

Review the generated questions and compare with the outputs from Experiments 11b and 11c to understand how chunk size affects question quality and diversity.

In [ ]:
print("=== Experiment 11a Summary ===")
print(f"Dataset        : Behind the Tech Podcast Transcripts")
print(f"Chunk size     : {CHUNK_SIZE} tokens (small retrieval chunks)")
print(f"Chunk overlap  : {CHUNK_OVERLAP} tokens")
print(f"Episodes loaded: {len(clustered_sample.documents)}")
print(f"Total chunks   : {len(clustered_sample.text_units)}")
print(f"Sampled chunks : {len(clustered_sample.sample_texts)}")
print(f"Data-local Qs  : {len(data_local_results.selected_questions)}")
print(f"Data-global Qs : {len(data_global_results.selected_questions)}")
print()
print("Sample data-local questions:")
for i, q in enumerate(data_local_results.selected_questions[:3]):
    print(f"  {i + 1}. {q.text}")
print()
print("Sample data-global questions:")
for i, q in enumerate(data_global_results.selected_questions[:3]):
    print(f"  {i + 1}. {q.text}")